In [0]:
print("Testing through databricks for git")

In [ ]:
print("This is a test for git integration with databricks")

In [ ]:
print("klayan - testing - printing")

# File-based data pipeline: Bronze -> Silver -> Gold

Run the cells from top to bottom. This example uses only Python's standard library. The landing CSV includes deliberately messy and invalid rows so the silver transformations can be seen in action.

In [ ]:
from pathlib import Path
import csv
import shutil

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
LANDING_DIR = DATA_DIR / "landing"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

LANDING_FILE = LANDING_DIR / "orders.csv"
BRONZE_FILE = BRONZE_DIR / "orders.csv"
SILVER_FILE = SILVER_DIR / "orders.csv"
REJECTED_FILE = SILVER_DIR / "rejected_orders.csv"
GOLD_FILE = GOLD_DIR / "daily_sales.csv"

LANDING_DIR.mkdir(parents=True, exist_ok=True)
sample_rows = [
    ["order_id", "order_date", "customer_id", "product", "category", "quantity", "unit_price", "region"],
    ["1001", "2026-01-02", "C001", " laptop ", " electronics ", "1", "1200", "north"],
    ["1002", "01/02/2026", " C002 ", "wireless mouse", "accessories", "2", "25.50", "SOUTH"],
    ["1003", "2026-01-03", "C001", "USB-C hub", "electronics", "1", "49.99", "North"],
    ["1004", "2026/01/03", "C003", "office chair", "furniture", "1", "189.99", "west"],
    ["1002", "2026-01-02", "C002", "wireless mouse", "accessories", "2", "25.50", "south"],
    ["", "2026-01-04", "C004", "notebook", "stationery", "3", "4.50", "east"],
    ["1005", "2026-01-04", "C004", "notebook", "stationery", "-2", "4.50", "east"],
    ["1006", "not-a-date", "C005", "desk lamp", "furniture", "1", "35.00", "west"],
]
with LANDING_FILE.open("w", encoding="utf-8", newline="") as file:
    csv.writer(file).writerows(sample_rows)

print(f"Sample source data: {LANDING_FILE}")

## Bronze layer

Copy the landing file unchanged to bronze to preserve the raw input.

In [ ]:
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(LANDING_FILE, BRONZE_FILE)
print(f"Bronze file written: {BRONZE_FILE}")

## Silver layer

Normalize strings and dates, validate quantities and prices, calculate line totals, deduplicate order IDs, and save rejected rows with a reason.

In [ ]:
from datetime import datetime
from decimal import Decimal, InvalidOperation
import json

REQUIRED_COLUMNS = {
    "order_id", "order_date", "customer_id", "product", "category",
    "quantity", "unit_price", "region",
}

def clean_order(raw):
    order_id = raw.get("order_id", "").strip().upper()
    customer_id = raw.get("customer_id", "").strip().upper()
    product = raw.get("product", "").strip().title()
    category = raw.get("category", "").strip().title()
    region = raw.get("region", "").strip().upper()
    if not all((order_id, customer_id, product, category, region)):
        raise ValueError("required text field is empty")

    date_value = raw.get("order_date", "").strip()
    parsed_date = None
    for date_format in ("%Y-%m-%d", "%m/%d/%Y", "%Y/%m/%d"):
        try:
            parsed_date = datetime.strptime(date_value, date_format).date()
            break
        except ValueError:
            continue
    if parsed_date is None:
        raise ValueError("order_date must be YYYY-MM-DD, MM/DD/YYYY, or YYYY/MM/DD")

    try:
        quantity = int(raw.get("quantity", "").strip())
    except ValueError as error:
        raise ValueError("quantity must be a whole number") from error
    if quantity <= 0:
        raise ValueError("quantity must be greater than zero")

    try:
        unit_price = Decimal(raw.get("unit_price", "").strip().replace("$", "").replace(",", ""))
    except InvalidOperation as error:
        raise ValueError("unit_price must be a valid amount") from error
    if not unit_price.is_finite() or unit_price < 0:
        raise ValueError("unit_price must be a non-negative amount")

    return {
        "order_id": order_id,
        "order_date": parsed_date.isoformat(),
        "customer_id": customer_id,
        "product": product,
        "category": category,
        "quantity": quantity,
        "unit_price": f"{unit_price:.2f}",
        "region": region,
        "line_total": f"{unit_price * quantity:.2f}",
    }

SILVER_DIR.mkdir(parents=True, exist_ok=True)
accepted = []
rejected = []
seen_order_ids = set()
with BRONZE_FILE.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    if reader.fieldnames is None:
        raise ValueError(f"CSV file has no header: {BRONZE_FILE}")
    missing_columns = REQUIRED_COLUMNS.difference(reader.fieldnames)
    if missing_columns:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing_columns))}")
    for source_row, raw in enumerate(reader, start=2):
        try:
            clean = clean_order(raw)
            if clean["order_id"] in seen_order_ids:
                raise ValueError("duplicate order_id")
            seen_order_ids.add(clean["order_id"])
            accepted.append(clean)
        except ValueError as error:
            rejected.append({
                "source_row": source_row,
                "reason": str(error),
                "raw_record": json.dumps(raw, sort_keys=True),
            })

silver_columns = [
    "order_id", "order_date", "customer_id", "product", "category",
    "quantity", "unit_price", "region", "line_total",
]
with SILVER_FILE.open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=silver_columns)
    writer.writeheader()
    writer.writerows(accepted)

with REJECTED_FILE.open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=["source_row", "reason", "raw_record"])
    writer.writeheader()
    writer.writerows(rejected)

print(f"Silver: {len(accepted)} accepted; rejected: {len(rejected)}")
print(f"Silver file: {SILVER_FILE}")
print(f"Rejected rows: {REJECTED_FILE}")

## Gold layer

Aggregate the clean silver records by date, category, and region.

In [ ]:
totals = {}
with SILVER_FILE.open("r", encoding="utf-8", newline="") as source:
    for row in csv.DictReader(source):
        key = (row["order_date"], row["category"], row["region"])
        group = totals.setdefault(key, {
            "order_count": 0,
            "quantity_sold": 0,
            "revenue": Decimal("0.00"),
        })
        group["order_count"] += 1
        group["quantity_sold"] += int(row["quantity"])
        group["revenue"] += Decimal(row["line_total"])

GOLD_DIR.mkdir(parents=True, exist_ok=True)
gold_columns = [
    "order_date", "category", "region", "order_count", "quantity_sold", "revenue",
]
with GOLD_FILE.open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=gold_columns)
    writer.writeheader()
    for (order_date, category, region), group in sorted(totals.items()):
        writer.writerow({
            "order_date": order_date,
            "category": category,
            "region": region,
            "order_count": group["order_count"],
            "quantity_sold": group["quantity_sold"],
            "revenue": f"{group['revenue']:.2f}",
        })

print(f"Gold file: {GOLD_FILE} ({len(totals)} groups)")

## Run and inspect the pipeline

Re-run the bronze, silver, and gold code cells in order whenever you want to rebuild the outputs from the landing sample.

In [ ]:
for label, output_file in (
    ("BRONZE", BRONZE_FILE),
    ("SILVER", SILVER_FILE),
    ("REJECTED", REJECTED_FILE),
    ("GOLD", GOLD_FILE),
):
    print(f"\n--- {label} ---")
    with output_file.open("r", encoding="utf-8", newline="") as file:
        print(file.read())